# 🧠🤖 第4周·Day5 — RAG实战架构设计与优化策略

> **学习目标**：理解RAG系统的完整架构设计，掌握检索→生成各阶段的优化方法

---


## 📖 核心知识点

### 1. RAG不是简单拼接，而是端到端优化
RAG系统的优化要分阶段进行：
- **检索质量** → 召回率、相关性
- **上下文质量** → 去重、排序、长度控制
- **生成质量** → 约束提示、后处理校验

### 2. 优化关键指标
| 指标 | 含义 | 目标值 |
|------|------|--------|
| 召回率 | 相关文档被检索到的比例 | > 0.9 |
| 相关性 | Top-K结果与问题的相关程度 | > 0.8 |
| 准确率 | 最终回答的正确性 | > 0.85 |
| 去重率 | 重复文档的消除比例 | > 0.7 |


### 3. 优化后的RAG架构
```
用户问题 → 预处理 → 多路检索 → 混合检索 → 重排序 → 去重 → 上下文优化 → LLM生成 → 后处理
```

💡 **关键改进**：
- 多路检索：同时使用向量、关键词、知识图谱
- 混合检索：BM25 + 向量相似度
- 动态上下文：根据问题复杂度调整上下文长度
- 生成约束：限制模型只能基于检索内容回答


In [ ]:
# 模拟RAG系统优化前后的对比
import pandas as pd
import random

random.seed(42)

# 模拟优化前后的检索结果
metrics_before = {
    '召回率': [0.65, 0.60, 0.70, 0.68, 0.62],
    '相关性': [0.55, 0.60, 0.58, 0.52, 0.65],
    '准确率': [0.50, 0.55, 0.48, 0.60, 0.52],
    '去重率': [0.30, 0.25, 0.35, 0.28, 0.32]
}

metrics_after = {
    '召回率': [0.92, 0.88, 0.95, 0.90, 0.87],
    '相关性': [0.85, 0.90, 0.82, 0.88, 0.86],
    '准确率': [0.88, 0.85, 0.90, 0.87, 0.86],
    '去重率': [0.75, 0.80, 0.72, 0.78, 0.76]
}

df_before = pd.DataFrame(metrics_before)
df_after = pd.DataFrame(metrics_after)

print('📊 优化前指标均值：')
print(df_before.mean())
print('')
print('📊 优化后指标均值：')
print(df_after.mean())
print('')
print('📈 提升幅度：')
print(((df_after.mean() - df_before.mean()) / df_before.mean() * 100).round(1))


## 🔧 分块策略对比

不同的文档分块方式直接影响检索质量：
- **固定大小分块**：简单但不考虑语义边界
- **语义分块**：按段落/句子边界切分，质量更高
- **滑动窗口**：块之间有重叠，避免信息遗漏


In [ ]:
# 文档分块策略演示
import re

sample_text = """
RAG（检索增强生成）是一种结合信息检索和文本生成的技术。它的核心思想是：
先从知识库中检索相关文档，再将这些文档作为上下文输入给大语言模型，
让模型基于真实信息生成回答。这种方法可以有效减少大模型的幻觉问题。
RAG系统主要由三个组件构成：检索器、知识库和生成器。
检索器负责从知识库中找到与用户问题最相关的文档片段。
知识库存储了大量的文档和知识信息。
生成器则基于检索到的内容生成最终答案。
"""

def fixed_chunk(text, size=50, overlap=10):
    """固定大小分块"""
    chunks = []
    for i in range(0, len(text), size - overlap):
        chunks.append(text[i:i+size])
    return chunks

def semantic_chunk(text):
    """按句子边界分块"""
    sentences = re.split(r'[。！？\n]', text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences

print('=== 固定大小分块 (size=50, overlap=10) ===')
for i, chunk in enumerate(fixed_chunk(sample_text.replace('\n',''), 50, 10)):
    print(f'Chunk {i+1}: {chunk[:40]}...')

print('\n=== 语义分块（按句子） ===')
for i, chunk in enumerate(semantic_chunk(sample_text)):
    print(f'Sentence {i+1}: {chunk[:50]}')


In [ ]:
# 可视化不同分块策略的覆盖质量
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

strategies = ['Fixed Chunk\n(No Overlap)', 'Fixed Chunk\n(With Overlap)', 'Semantic\nChunk']
recall = [0.62, 0.78, 0.91]
precision = [0.70, 0.75, 0.88]

x = np.arange(len(strategies))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, recall, width, label='Recall', color='#4ECDC4')
bars2 = ax.bar(x + width/2, precision, width, label='Precision', color='#FF6B6B')

ax.set_ylabel('Score')
ax.set_title('Chunking Strategy Comparison', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(strategies)
ax.legend()
ax.set_ylim(0, 1.1)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=11)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.savefig('/root/learning-notebooks/第4周/day5_chunking.png', dpi=150)
print('Chart saved to day5_chunking.png')


## 🔍 重排序（Reranking）优化

重排序是提升检索质量的"杀手锏"：
1. 初筛（召回阶段）：用向量检索快速召回 Top-50
2. 精排（重排序阶段）：用 Cross-Encoder 对 Top-50 重新打分
3. 最终取 Top-K 送给 LLM


In [ ]:
# 模拟重排序过程
import numpy as np

# 模拟初筛结果（向量检索召回Top-10）
docs = [
    'Transformer是一种基于自注意力机制的神经网络架构',
    '自注意力机制允许模型关注输入序列中的所有位置',
    '苹果公司发布了新款iPhone手机',
    'BERT是Google开发的双向预训练语言模型',
    '位置编码解决了Transformer中位置信息缺失的问题',
    '今天天气不错适合出门散步',
    'Multi-Head Attention通过并行计算多个注意力头',
    '张三和李四是好朋友经常一起打篮球',
    '残差连接帮助深层网络避免梯度消失问题',
    '超市今天打折蔬菜水果都很便宜'
]

query = 'Transformer的核心技术是什么？'

# 模拟向量检索分数（初筛）
vector_scores = np.array([0.92, 0.88, 0.25, 0.78, 0.82, 0.12, 0.75, 0.10, 0.70, 0.15])

# 模拟Cross-Encoder重排序分数（精排）
rerank_scores = np.array([0.95, 0.90, 0.20, 0.85, 0.88, 0.08, 0.82, 0.05, 0.80, 0.12])

# 对比排序结果
vector_rank = np.argsort(-vector_scores)[:5]
rerank_rank = np.argsort(-rerank_scores)[:5]

print('🔍 向量检索 Top-5：')
for rank, idx in enumerate(vector_rank):
    print(f'  {rank+1}. [{vector_scores[idx]:.2f}] {docs[idx][:35]}...')

print('\n⚡ 重排序后 Top-5：')
for rank, idx in enumerate(rerank_rank):
    print(f'  {rank+1}. [{rerank_scores[idx]:.2f}] {docs[idx][:35]}...')

print(f'\n排名变化：文档3从第2升到第2（分数提升{rerank_scores[1]-vector_scores[1]:.2f}）')


In [ ]:
# 可视化重排序效果对比
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 向量检索分数
top5_v = np.argsort(-vector_scores)[:5]
colors_v = ['#4ECDC4' if s > 0.7 else '#FF6B6B' for s in vector_scores[top5_v]]
ax1.barh(range(5), vector_scores[top5_v][::-1], color=colors_v[::-1])
ax1.set_yticks(range(5))
ax1.set_yticklabels([f'Doc {i}' for i in top5_v[::-1]+1])
ax1.set_xlabel('Score')
ax1.set_title('Vector Retrieval (Top-5)')

# 重排序分数
top5_r = np.argsort(-rerank_scores)[:5]
colors_r = ['#4ECDC4' if s > 0.8 else '#FF6B6B' for s in rerank_scores[top5_r]]
ax2.barh(range(5), rerank_scores[top5_r][::-1], color=colors_r[::-1])
ax2.set_yticks(range(5))
ax2.set_yticklabels([f'Doc {i}' for i in top5_r[::-1]+1])
ax2.set_xlabel('Score')
ax2.set_title('After Re-ranking (Top-5)')

plt.tight_layout()
plt.savefig('/root/learning-notebooks/第4周/day5_reranking.png', dpi=150)
print('Chart saved to day5_reranking.png')


## 📏 上下文窗口管理

LLM的上下文窗口有限，如何高效利用？
- **硬截断**：超过长度直接截断（简单但丢失信息）
- **摘要压缩**：对检索文档做摘要再输入（保留核心信息）
- **动态裁剪**：根据问题相关度动态调整每个文档的长度


In [ ]:
# 上下文窗口管理策略模拟
context_limit = 2000  # 模拟2000 token的上下文窗口

# 模拟检索到的文档及其相关性分数
retrieved_docs = [
    {'id': 1, 'text': 'A' * 800, 'score': 0.95},
    {'id': 2, 'text': 'B' * 600, 'score': 0.90},
    {'id': 3, 'text': 'C' * 500, 'score': 0.85},
    {'id': 4, 'text': 'D' * 400, 'score': 0.80},
    {'id': 5, 'text': 'E' * 300, 'score': 0.75},
]

total_tokens = sum(len(d['text']) for d in retrieved_docs)
print(f'检索文档总长度: {total_tokens} tokens')
print(f'上下文窗口限制: {context_limit} tokens')
print(f'超出限制: {total_tokens - context_limit} tokens')

# 动态裁剪策略
remaining = context_limit
selected = []
for doc in sorted(retrieved_docs, key=lambda x: x['score'], reverse=True):
    if remaining <= 0:
        break
    alloc = min(len(doc['text']), remaining)
    selected.append({
        'id': doc['id'],
        'original_len': len(doc['text']),
        'allocated_len': alloc,
        'score': doc['score']
    })
    remaining -= alloc

print('\n📋 动态裁剪分配方案：')
for s in selected:
    pct = s['allocated_len'] / s['original_len'] * 100
    print(f"  Doc {s['id']}: 分配 {s['allocated_len']}/{s['original_len']} ({pct:.0f}%) [score={s['score']:.2f}]")


## 🧪 课堂练习（5分钟）

1. 什么是RAG的"检索瓶颈"？
2. 上下文窗口大小对RAG有什么影响？
3. 如何解决RAG的"幻觉"问题？

💡 提示：从检索质量、上下文质量、生成质量三方面思考


## 📝 课后测试

❶ RAG系统中最常用的向量数据库是？
A. MySQL  B. Chroma  C. PostgreSQL  D. Redis

❷ 以下哪种方法能有效提升RAG检索质量？
A. 增加上下文窗口  B. 使用单一检索器  C. 多路检索融合  D. 减少文档分块大小

❸ GraphRAG相比传统RAG的优势是？
A. 检索速度更快  B. 能够理解文档内部逻辑关系  C. 占用内存更小  D. 支持更多文档格式

❹ RAG系统中的"去重"主要是为了解决什么问题？
A. 重复计算  B. 上下文冗余  C. 内存占用  D. 检索延迟

❺ 企业级RAG系统最重要的优化目标是什么？
A. 检索速度  B. 回答准确性  C. 系统成本  D. 用户体验

回复答案帮你批改 ✅


## 🔑 今日英文术语

- **Retrieval-Augmented Generation** [rɪˈtrivəl ˌɔːɡˈmentɪd ˈdʒenəˈreɪʃən] 检索增强生成
- **Context Window** [ˈkɒntekst ˈwɪndəʊ] 上下文窗口
- **Chunking Strategy** [ˈtʃʌŋkɪŋ ˈstrætədʒi] 分块策略
- **Re-ranking** [ˌriːˈræŋkɪŋ] 重排序
- **Precision-Recall Trade-off** [prɪˈsɪʒən rɪˈkɔːl ˈtreɪd-ɒf] 精确度-召回率权衡


## 🎬 推荐视频

【RAG系统优化实战】（45分钟）
https://www.bilibili.com/video/BV1xt421C7sP

## 📖 延伸阅读

【企业级RAG架构设计指南】
https://arxiv.org/abs/2401.15884
